# 03 · Sliding windows

Phase 1, Nov 8–9. The first model (notebook 02) only looked at the labelled gesture part. The wearable has no labels, so here the model looks at **every 30-reading window**, stepping 5 readings at a time, along the whole recording, just as it will on the live sensor.

**Window labels:** a window is *BFRB-like* if at least half of it falls in the gesture part of a BFRB-like recording. Everything else is *ordinary*: ordinary gestures, and the hand moving into place before any gesture.

In [ ]:
import sys
sys.path.append("..")
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

from saha.features import ACC, ROT, FEATURES, WINDOW, STEP, add_channels, summarise, window_starts

DATA = "../data/cmi-detect-behavior-with-sensor-data"
LABELS = ["sequence_id", "subject", "sequence_type", "orientation", "phase", "gesture"]
df = pd.read_csv(f"{DATA}/train.csv", usecols=LABELS + ACC + ROT)

## 1. Cut every recording into windows

In [ ]:
rows = []
for sid, seq in df.groupby("sequence_id", sort=False):
    g = add_channels(seq.reset_index(drop=True))
    target = g.sequence_type.iloc[0] == "Target"
    in_gesture = (g.phase == "Gesture").to_numpy()
    for start in window_starts(len(g)):
        w = g.iloc[start:start + WINDOW]
        feats = summarise(w).to_dict()
        feats.update(
            sequence_id=sid, subject=g.subject.iloc[0], sequence_type=g.sequence_type.iloc[0],
            orientation=g.orientation.iloc[0], gesture=g.gesture.iloc[0], start=start,
            label=int(target and in_gesture[start:start + WINDOW].mean() >= 0.5),
            touches_gesture=bool(in_gesture[start:start + WINDOW].any()),
        )
        rows.append(feats)

win = pd.DataFrame(rows)
X, y, groups = win[FEATURES], win.label, win.subject
print(f"{len(win)} windows from {win.sequence_id.nunique()} recordings; {y.mean():.1%} are BFRB-like")

## 2. Test on unseen people

Same 5-fold split by person as notebook 02.

In [ ]:
def new_model():
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
prob = np.zeros(len(win))
for train, test in cv.split(X, y, groups):
    assert not set(groups.iloc[train]) & set(groups.iloc[test])
    prob[test] = new_model().fit(X.iloc[train], y.iloc[train]).predict_proba(X.iloc[test])[:, 1]
win["prob"] = prob
win["pred"] = (prob >= 0.5).astype(int)

print("Per window:")
print(f"  balanced accuracy: {balanced_accuracy_score(y, win.pred):.3f}")
print(f"  F1 for BFRB-like:  {f1_score(y, win.pred):.3f}")
print(f"  ROC AUC:           {roc_auc_score(y, prob):.3f}")
ConfusionMatrixDisplay(confusion_matrix(y, win.pred), display_labels=["ordinary", "BFRB-like"]).plot(cmap="Blues", colorbar=False)
plt.title("Windows, unseen people");

## 3. Would it alert at the right times?

This is closer to what the user feels. For each recording:

- **BFRB-like recording:** did an alert fire during the gesture part? (caught)
- **Ordinary recording:** did any window at all trigger an alert? (false alarm)

An alert fires when **2 windows in a row** are called BFRB-like, which filters out single noisy windows.

In [ ]:
def alerts(p, threshold=0.5, in_a_row=2):
    hit = (p >= threshold).astype(int)
    return np.convolve(hit, np.ones(in_a_row, int), "valid") >= in_a_row if len(hit) >= in_a_row else np.zeros(1, bool)

def recording_result(r, threshold=0.5):
    fired = alerts(r.prob.to_numpy(), threshold)
    if r.sequence_type.iloc[0] == "Target" and len(r) >= 2:
        # an alert counts as a catch only if its second window overlaps the gesture part
        return (fired & r.touches_gesture.to_numpy()[1:]).any()
    return fired.any()

for t in (0.5, 0.6, 0.7, 0.8):
    res = win.groupby("sequence_id").apply(lambda r: recording_result(r, t), include_groups=False)
    kind = win.groupby("sequence_id").sequence_type.first()
    caught = res[kind == "Target"].mean()
    false_alarm = res[kind == "Non-Target"].mean()
    print(f"threshold {t}:  BFRB-like caught {caught:.1%}   ordinary with a false alarm {false_alarm:.1%}")

Which ordinary gestures cause false alarms (threshold 0.5)?

In [ ]:
res = win.groupby("sequence_id").apply(recording_result, include_groups=False).rename("alerted")
seq_info = win.groupby("sequence_id")[["sequence_type", "gesture"]].first().join(res)
(seq_info.groupby(["sequence_type", "gesture"]).alerted.mean().mul(100).round(1)
     .sort_values(ascending=False).rename("% of recordings alerted").to_frame())

## 4. One recording, window by window

In [ ]:
example = win[(win.gesture == "Forehead - pull hairline") & (win.orientation == "Seated Straight")].sequence_id.iloc[0]
r = win[win.sequence_id == example]
plt.figure(figsize=(9, 3))
plt.step(r.start, r.prob, where="post", label="model's BFRB-like probability")
plt.step(r.start, r.label, where="post", linestyle="--", label="true label")
plt.axhline(0.5, color="grey", lw=0.8)
plt.xlabel("window start (reading)"); plt.ylim(-0.05, 1.05); plt.legend(loc="upper left")
plt.title(f"{example}: Forehead - pull hairline");

## 5. Save a model for Phase 3

Trained on **all** CMI people. The scores above are the honest ones; this saved model is what will run on the wearable's data. `models/` is not committed to Git.

In [ ]:
Path("../models").mkdir(exist_ok=True)
final = new_model().fit(X, y)
joblib.dump({"model": final, "features": FEATURES, "window": WINDOW, "step": STEP}, "../models/window_model.joblib")
print("saved ../models/window_model.joblib")

## 6. Score log

| Date | Change | Balanced accuracy | ROC AUC |
|---|---|---|---|
| Nov 1 | 02: gesture part only, 36 features, unseen people | 0.886 | 0.957 |
| Nov 1 | 03: sliding 30-reading windows (step 5), unseen people, per window | 0.863 | 0.957 |

## Limits

- Windows are in **readings**, not seconds, because CMI has no timestamps. Once the CMI sample rate is confirmed, the wearable has to be resampled to match (see HARDWARE.md).
- In CMI, every recording is a deliberate, acted gesture. Real life has much more ordinary movement, so false alarms will be more common on the wearable.